# Data Preprocessing and Feature Engineering - Adult Dataset

In [1]:

import pandas as pd

df = pd.read_csv("adult_with_headers (1).csv")
df.head()


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### Handling Missing Values

In [2]:

df = df.replace(" ?", pd.NA)

for col in ["workclass", "occupation", "native_country"]:
    df[col] = df[col].fillna(df[col].mode()[0])

df.isnull().sum()


age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64

### Separating Columns

In [3]:

num_cols = ['age','fnlwgt','education_num','capital_gain','capital_loss','hours_per_week']

cat_cols = ['workclass','education','marital_status','occupation',
            'relationship','race','sex','native_country','income']


### Scaling

In [4]:

from sklearn.preprocessing import MinMaxScaler

df_mm = df.copy()

scaler = MinMaxScaler()
df_mm[num_cols] = scaler.fit_transform(df_mm[num_cols])

df_mm.head()


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,0.301370,State-gov,0.044302,Bachelors,0.800000,Never-married,Adm-clerical,Not-in-family,White,Male,0.02174,0.0,0.397959,United-States,<=50K
1,0.452055,Self-emp-not-inc,0.048238,Bachelors,0.800000,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.00000,0.0,0.122449,United-States,<=50K
2,0.287671,Private,0.138113,HS-grad,0.533333,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.00000,0.0,0.397959,United-States,<=50K
3,0.493151,Private,0.151068,11th,0.400000,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0.00000,0.0,0.397959,United-States,<=50K
4,0.150685,Private,0.221488,Bachelors,0.800000,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0.00000,0.0,0.397959,Cuba,<=50K


### Encoding

In [5]:

df_encoded = pd.get_dummies(df_mm, columns=['sex','income','race'], drop_first=True)

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in ['workclass','education','marital_status','occupation','relationship','native_country']:
    df_encoded[col] = le.fit_transform(df_encoded[col])

df_encoded = df_encoded.astype(int)
df_encoded.head()


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,capital_gain,capital_loss,hours_per_week,native_country,sex_ Male,income_ >50K,race_ Asian-Pac-Islander,race_ Black,race_ Other,race_ White
0,0,6,0,9,0,4,0,1,0,0,0,38,1,0,0,0,0,1
1,0,5,0,9,0,2,3,0,0,0,0,38,1,0,0,0,0,1
2,0,3,0,11,0,0,5,1,0,0,0,38,1,0,0,0,0,1
3,0,3,0,1,0,2,5,0,0,0,0,38,1,0,0,1,0,0
4,0,3,0,9,0,2,9,5,0,0,0,4,0,0,0,1,0,0


### Feature Engineering

In [6]:

df_encoded["capital_diff"] = df_encoded["capital_gain"] - df_encoded["capital_loss"]
df_encoded["work_intensity"] = df_encoded["hours_per_week"] * df_encoded["education_num"]

df_encoded.head()


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,capital_gain,capital_loss,hours_per_week,native_country,sex_ Male,income_ >50K,race_ Asian-Pac-Islander,race_ Black,race_ Other,race_ White,capital_diff,work_intensity
0,0,6,0,9,0,4,0,1,0,0,0,38,1,0,0,0,0,1,0,0
1,0,5,0,9,0,2,3,0,0,0,0,38,1,0,0,0,0,1,0,0
2,0,3,0,11,0,0,5,1,0,0,0,38,1,0,0,0,0,1,0,0
3,0,3,0,1,0,2,5,0,0,0,0,38,1,0,0,1,0,0,0,0
4,0,3,0,9,0,2,9,5,0,0,0,4,0,0,0,1,0,0,0,0


### Log Transformation

In [7]:

import numpy as np

df_encoded["capital_gain"] = np.log1p(df_encoded["capital_gain"])


### Isolation Forest

In [8]:

from sklearn.ensemble import IsolationForest

iso = IsolationForest(contamination=0.05, random_state=42)
df_encoded["outlier"] = iso.fit_predict(df_encoded)

df_encoded["outlier"].value_counts()


outlier
 1    30933
-1     1628
Name: count, dtype: int64

### Outlier Detection using Isolation Forest

Isolation Forest was used to detect outliers in the dataset.  
The algorithm assigns -1 to outliers and 1 to normal observations.

A small percentage of data points were identified as outliers, which may represent unusual or extreme cases in the dataset.

In [10]:
!pip install ppscore

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for ppscore: filename=ppscore-1.3.1-py2.py3-none-any.whl size=13262 sha256=da6351cc073a81879a42c9984822f47ff62f26a3dc5c194c58f4039cf1af61af
  Stored in directory: c:\users\rajur\appdata\local\pip\cache\wheels\b4\95\1f\bfb3b7b3845232909c01caa738277f86d0148164d3a617ad42
Successfully built ppscore


In [11]:
import ppscore as pps

pps_matrix = pps.matrix(df_encoded)
pps_matrix.head()

,x,y,ppscore,case,is_valid_score,metric,baseline_score,model_score,model
0,age,age,1.0,predict_itself,True,None,0.0000,1.000000,None
1,age,workclass,0.0,regression,True,mean absolute error,0.5146,0.587713,DecisionTreeRegressor()
2,age,fnlwgt,0.0,target_is_constant,True,None,1.0000,1.000000,None
3,age,education,0.0,regression,True,mean absolute error,2.7590,2.931496,DecisionTreeRegressor()
4,age,education_num,0.0,regression,True,mean absolute error,0.0146,0.028777,DecisionTreeRegressor()


In [12]:
pps_matrix[pps_matrix['y'] == 'income_ >50K'].sort_values(by='ppscore', ascending=False)

,x,y,ppscore,case,is_valid_score,metric,baseline_score,model_score,model
286,income_ >50K,income_ >50K,1.0,predict_itself,True,None,0.0000,1.000000,None
13,age,income_ >50K,0.0,regression,True,mean absolute error,0.2424,0.367187,DecisionTreeRegressor()
244,native_country,income_ >50K,0.0,regression,True,mean absolute error,0.2424,0.362267,DecisionTreeRegressor()
412,work_intensity,income_ >50K,0.0,regression,True,mean absolute error,0.2424,0.367369,DecisionTreeRegressor()
391,capital_diff,income_ >50K,0.0,regression,True,mean absolute error,0.2424,0.361830,DecisionTreeRegressor()
370,race_ White,income_ >50K,0.0,regression,True,mean absolute error,0.2424,0.364089,DecisionTreeRegressor()
349,race_ Other,income_ >50K,0.0,regression,True,mean absolute error,0.2424,0.367122,DecisionTreeRegressor()
328,race_ Black,income_ >50K,0.0,regression,True,mean absolute error,0.2424,0.363812,DecisionTreeRegressor()
307,race_ Asian-Pac-Islander,income_ >50K,0.0,regression,True,mean absolute error,0.2424,0.367388,DecisionTreeRegressor()
265,sex_ Male,income_ >50K,0.0,regression,True,mean absolute error,0.2424,0.348365,DecisionTreeRegressor()


### PPS Score Analysis

PPS score was used to measure the predictive power of each feature for the target variable.

Features with higher PPS score have stronger relationship with income and are more important for prediction.

### Conclusion

In this assignment, data preprocessing and feature engineering techniques were applied including handling missing values, scaling, encoding, and feature transformation.

Outliers were detected using Isolation Forest, and feature importance was analyzed using PPS score.

These steps improve data quality and help build better machine learning models.